In [1]:
from notebookutils import mssparkutils
import re

# === IDs del Lakehouse adjunto al notebook (robusto) ===
WS_ID = spark.conf.get("trident.workspace.id")
LH_ID = spark.conf.get("trident.lakehouse.id")

display(WS_ID,LH_ID)

StatementMeta(, 40116d0a-a194-443e-b092-4e0383f6b633, 3, Finished, Available, Finished, False)

'3ef6ca27-57c7-40cd-9ca0-d878f9ddc003'

In [5]:
from notebookutils import mssparkutils
import re

# === IDs del Lakehouse adjunto al notebook (robusto) ===
WS_ID = spark.conf.get("trident.workspace.id")
LH_ID = spark.conf.get("trident.lakehouse.id")

#display(LH_ID)

# Folder donde están los parquets
source_folder = f"abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

# === DESTINO ===
WAREHOUSE_NAME = "sqltranmigration"   
TARGET_SCHEMA  = "silver"        # schema en el Warehouse
WRITE_MODE     = "append"        # "append" o "overwrite"

def file_to_table_name(file_name: str) -> str:
    if not re.match(r"^silver_.+\.parquet$", file_name, flags=re.IGNORECASE):
        raise ValueError(f"Nombre de archivo inesperado: {file_name}")
    base = re.sub(r"^silver_", "", file_name, flags=re.IGNORECASE)
    base = re.sub(r"\.parquet$", "", base, flags=re.IGNORECASE)
    base = base.strip().replace(" ", "_").replace("-", "_")
    base = re.sub(r"_+", "_", base).strip("_")
    return base.lower()

# 1) Listar parquets
items = mssparkutils.fs.ls(source_folder)
parquet_files = [i for i in items if i.name.lower().endswith(".parquet")]

print(f"Parquets encontrados en {source_folder}: {len(parquet_files)}")
for f in parquet_files:
    print(" -", f.name)

# 2) Cargar uno por uno
for f in parquet_files:
    src_path = f.path
    table = file_to_table_name(f.name)
    target = f"{WAREHOUSE_NAME}.{TARGET_SCHEMA}.{table}"

    print(f"\n==> {f.name}  -->  {target}")

    df = spark.read.parquet(src_path)

    # Escribir a Warehouse (la tabla debe existir)
    #df.write.mode(WRITE_MODE).synapsesql(target)

    print(f"OK: {target}")


StatementMeta(, 40116d0a-a194-443e-b092-4e0383f6b633, 7, Finished, Available, Finished, False)

Parquets encontrados en abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/: 0
